# ResNet50 Stage 3 — D3 External Validation Only

**Frozen-checkpoint evaluation on the locked RealWaste D3 Section 2 cohort.** It loads the five seed-matched standalone ResNet50 checkpoints (`ResNet50_Torchvision_seed*.pt`) and evaluates the same 3,497 D3 images for each seed. D1 original-test evaluation is not performed.

## 1. Configuration and inputs

Uses the user-attached Stage 3 checkpoint dataset and the D3 Section 2 RealWaste manifest. Loads the 5 seed-matched standalone ResNet50 checkpoints from `Section3/Classification_checkpoints/Resnet_50/` and evaluates the locked external cohort only. No training, fine-tuning, or adaptation is performed.

In [1]:
from __future__ import annotations
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, traceback
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone

# Kaggle input location confirmed by the user; adjust ONLY if Kaggle moves the dataset.
INPUT_ROOT = Path('/kaggle/input/datasets/akash264208/manifist-part2/Section3')
OUTPUT = Path('/kaggle/working/ResNet50_D3_Standalone_External_Validation')
OUTPUT.mkdir(parents=True, exist_ok=True)
SEEDS = (7, 49, 112, 345, 476)
MODEL_NAME = 'ResNet50'
CHECKPOINT_MODEL_METADATA = 'ResNet-50'
CHECKPOINT_ARCHITECTURE = 'torchvision.resnet50'
CLASSES = ('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
EXPECTED_EXTERNAL_COUNT = 3497
EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT = '0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0'
# Set this to the exact Kaggle folder containing the D3 Section 2 CSV files.
# Example: '/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_D3_SECTION2_OUTPUTS'
# If None, the notebook locates unique matching files under /kaggle/input.
D3_MANIFEST_ROOT = '/kaggle/input/datasets/akash264208/manifist-part2/Section3/manifist'
BATCH_SIZE = 32

# Optional genuine historical Stage 2 standalone ResNet50 prediction files.
# Each file, when present, must have source_path and predicted_class_id columns,
# and preferably all six prob_CLASS columns. They must come from Stage 2.
# Set None to discover files automatically in the attached private Kaggle input.
HISTORICAL_REFERENCE_DIR = None
REQUIRE_HISTORICAL_REFERENCE = False  # Set True to STOP external evaluation if references are absent.

assert INPUT_ROOT.is_dir(), f'Input dataset was not attached: {INPUT_ROOT}'
print('Kaggle input:', INPUT_ROOT)
print('Output directory:', OUTPUT)
print('Seeds:', SEEDS)
print('Frozen standalone ResNet50 model; no training or fine-tuning.')

Kaggle input: /kaggle/input/datasets/akash264208/manifist-part2/Section3
Output directory: /kaggle/working/ResNet50_D3_Standalone_External_Validation
Seeds: (7, 49, 112, 345, 476)
Frozen standalone ResNet50 model; no training or fine-tuning.


## 2. Dependencies and strict standalone-checkpoint inventory

Load exactly 5 Stage 2 standalone ResNet50 checkpoints (`ResNet50_Torchvision_seed*.pt`) for seeds 7, 49, 112, 345, and 476. The REMNET ensemble constituent checkpoints are not used.

In [2]:
# Load dependencies without downloading pretrained weights.
for name in ('kagglehub', 'timm'):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', name])
import kagglehub, timm, re, numpy as np, pandas as pd, torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU inference selected')
print('torch:', torch.__version__, 'timm:', timm.__version__)

def only_directory_with_name(root, name):
    matches = sorted(p for p in root.rglob('*') if p.is_dir() and p.name.casefold() == name.casefold())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one directory {name!r} under {root}; found {matches}')
    return matches[0]

def unique_by_filename(root, filename):
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r} in {root}: {matches}')
    return matches[0]

CKPT_ROOT = only_directory_with_name(INPUT_ROOT, 'Classification_checkpoints')
REMNET_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Remnet')
STANDALONE_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Resnet_50')
RESULTS_ROOT = only_directory_with_name(INPUT_ROOT, 'per_seed_performance_summary')
# D3 Section 2 manifest and class mapping are read directly from the user's Kaggle input.
# There is no embedded or reconstructed manifest.
def d3_csv_path(filename):
    root = Path(D3_MANIFEST_ROOT) if D3_MANIFEST_ROOT else Path('/kaggle/input')
    if not root.is_dir():
        raise FileNotFoundError(f'D3 manifest folder not attached: {root}')
    return unique_by_filename(root, filename)

EXTERNAL_MANIFEST = d3_csv_path('D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv')
CLASS_MAPPING = d3_csv_path('D3_Section2_Class_Mapping.csv')

# Only match the original standalone naming pattern from Resnet_50_Acc_s2.ipynb.
CHECKPOINTS = {}
for seed in SEEDS:
    expected_name = f'ResNet50_Torchvision_seed{seed}.pt'
    matches = sorted(p for p in STANDALONE_CKPT_DIR.rglob('*.pt')
                     if re.fullmatch(rf'ResNet50_Torchvision_seed{seed}(?:\(\d+\))?\.pt', p.name, re.IGNORECASE))
    if len(matches) != 1:
        raise RuntimeError(f'Expected precisely one standalone ResNet50 checkpoint for seed {seed} '
                           f'(preferred filename {expected_name}); found: {matches}')
    CHECKPOINTS[seed] = matches[0]

all_pts = sorted(CKPT_ROOT.rglob('*.pt'))
standalone_pts = sorted(STANDALONE_CKPT_DIR.rglob('*.pt'))
remnet_pts = sorted(REMNET_CKPT_DIR.rglob('*.pt'))
assert len(standalone_pts) == 5, f'Expected 5 standalone ResNet50 checkpoints, found {len(standalone_pts)}'
assert len(remnet_pts) == 15, f'Expected 15 REMNET ensemble constituent checkpoints, found {len(remnet_pts)}'
assert len(all_pts) == 30, f'Expected 30 total Stage 2 checkpoints, found {len(all_pts)}'
assert not set(CHECKPOINTS.values()).intersection(remnet_pts)

# Stage 2 standalone ResNet50 historical references; do not use REMNET reports.
HISTORICAL_PER_SEED = next(iter(sorted(RESULTS_ROOT.rglob('ResNet50_Performance_Per_Seed.csv'))), None)
HISTORICAL_SUMMARY = next(iter(sorted(RESULTS_ROOT.rglob('ResNet50_5_Seed_Performance_Summary.csv'))), None)
print('Total .pt checkpoints:', len(all_pts), '| Standalone ResNet50:', len(standalone_pts),
      '| REMNET constituent checkpoints:', len(remnet_pts))
for seed, path in CHECKPOINTS.items(): print('STANDALONE ResNet50', seed, ':', path)
for path in [EXTERNAL_MANIFEST, CLASS_MAPPING]: print('CSV:', path)
print('Historical ResNet50 per-seed:', HISTORICAL_PER_SEED or 'not present')
print('Historical ResNet50 summary:', HISTORICAL_SUMMARY or 'not present')

GPU: Tesla T4
torch: 2.11.0+cu130 timm: 1.0.30
Total .pt checkpoints: 30 | Standalone ResNet50: 5 | REMNET constituent checkpoints: 15
STANDALONE ResNet50 7 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed7.pt
STANDALONE ResNet50 49 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed49.pt
STANDALONE ResNet50 112 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed112.pt
STANDALONE ResNet50 345 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed345.pt
STANDALONE ResNet50 476 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed476.pt
CSV: /kaggle/input/datasets/akash264208/manifist-part2/Section3/manifist/D3_Section2_Final_Automated_

## 3. Validate the locked D3 manifest and six-class RealWaste mapping

Validates the user-provided D3 Section 2 manifest, class mapping, image count, unique hashes, class indices, eligibility status, and locked external-cohort fingerprint.

In [3]:
def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for b in iter(lambda:stream.read(chunk_size),b''):
            h.update(b)
    return h.hexdigest()

def canonical_external_fingerprint(df):
    h = hashlib.sha256()
    for row in df.sort_values('external_relative_path').itertuples(index=False):
        h.update((row.external_relative_path+'|'+str(int(row.output_index))+'|'+
                  row.sha256+'\n').encode('utf-8'))
    return h.hexdigest()

external_df = pd.read_csv(EXTERNAL_MANIFEST, dtype=str, keep_default_na=False)
mapping_df = pd.read_csv(CLASS_MAPPING, dtype=str, keep_default_na=False)

assert set(['external_relative_path','class','output_index','sha256','final_status']) <= set(external_df.columns)
assert set(['d3_source_label','remnet_trained_class','remnet_output_index','include_for_primary_d3']) <= set(mapping_df.columns)
assert len(external_df)==EXPECTED_EXTERNAL_COUNT, 'Unexpected D3 manifest count (should be 3497).'
assert external_df['sha256'].nunique()==EXPECTED_EXTERNAL_COUNT, 'Duplicate hashes in external manifest.'
assert external_df['external_relative_path'].nunique()==EXPECTED_EXTERNAL_COUNT
assert set(external_df['class']) == set(CLASSES)
assert external_df['final_status'].eq('ELIGIBLE_AFTER_REFERENCE_VERIFICATION').all(), 'Noneligible images in frozen manifest.'
included_mapping = mapping_df.loc[mapping_df['include_for_primary_d3'].str.lower().eq('true')].copy()
assert len(included_mapping) == 6
assert set(included_mapping['remnet_trained_class']) == set(CLASSES)
for item in included_mapping.itertuples(index=False):
    assert int(item.remnet_output_index) == CLASS_TO_ID[item.remnet_trained_class]
    if 'class_review' in mapping_df.columns:
        assert item.class_review == 'CONFIRMED_BY_USER'
assert set(external_df['source_category']) == set(included_mapping['d3_source_label'])
assert all(int(i)==CLASS_TO_ID[c] for c,i in zip(external_df['class'],external_df['output_index']))

EXTERNAL_FINGERPRINT=canonical_external_fingerprint(external_df)
assert EXTERNAL_FINGERPRINT == EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT, (EXTERNAL_FINGERPRINT,'External cohort fingerprint mismatch')
print('D3 frozen manifest verified:',len(external_df),'Fingerprint:',EXTERNAL_FINGERPRINT)
print('Six class mapping:',CLASS_TO_ID)
print('External per-class counts:',external_df['class'].value_counts().sort_index().to_dict())

D3 frozen manifest verified: 3497 Fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0
Six class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
External per-class counts: {'cardboard': 454, 'glass': 419, 'metal': 757, 'paper': 500, 'plastic': 889, 'trash': 478}


## 4. Load the attached RealWaste D3 dataset

The notebook reads the RealWaste images from the mounted Kaggle input and uses the D3 Section 2 manifest to select and verify the 3,497 evaluation images. No D1 original-test evaluation is performed in this D3-only workflow.

In [4]:
# Stage 3 D3 dataset path — fixed mounted Kaggle input.
EXTERNAL_HANDLE = 'akash264208/real-waste'
external_download = Path('/kaggle/input/datasets/akash264208/real-waste')
external_download_method = 'attached Kaggle dataset'
assert external_download.is_dir(), f'D3 RealWaste dataset missing: {external_download}'
print('D3 RealWaste:', external_download)

D3 RealWaste: /kaggle/input/datasets/akash264208/real-waste


## 5. Resolve D3 images and verify every locked image hash

Resolve the 3,497 D3 manifest entries against the attached RealWaste image folders and verify each image's SHA-256 before inference.

In [5]:
IMG_EXT={'.jpg','.jpeg','.png','.webp','.bmp'}

def safe_parts(relative):
    p=PurePosixPath(str(relative).replace('\\','/'))
    if p.is_absolute() or not p.parts or any(s in ('','.','..') for s in p.parts):
        raise ValueError(f'Unsafe image relative path: {relative!r}')
    return p.parts

def external_class_roots(root):
    expected=set(included_mapping['d3_source_label'])
    candidates=[root] + sorted((p for p in root.rglob('*') if p.is_dir()), key=str)
    return [p for p in candidates if all((p/c).is_dir() for c in expected)]

def resolve_external(external,download):
    candidates=external_class_roots(download)
    print('D3 RealWaste class-root candidates:',candidates)
    if not candidates: raise FileNotFoundError('Missing RealWaste source-category directories.')
    expected_rows=[(safe_parts(r.external_relative_path),r.sha256,r.source_category)
                   for _,r in external.iterrows()]
    for root in candidates:
        try:
            dirs=[p for p in root.iterdir() if p.is_dir()]
            image_counts={p.name:sum(x.is_file() and x.suffix.lower() in IMG_EXT
                         for x in p.rglob('*')) for p in dirs}
            if len(image_counts)!=9 or sum(image_counts.values())!=4752:
                raise RuntimeError(f'Expected nine RealWaste folders / 4752 images: {image_counts}')
            if sum(image_counts[c] for c in included_mapping['d3_source_label'])!=3587:
                raise RuntimeError('D3 six-category source total differs from Section 2.')
            resolved=[]
            for parts,digest,source_cls in expected_rows:
                image=root.joinpath(*parts)
                if not image.is_file() or parts[0]!=source_cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'D3 image hash mismatch: {image}')
                resolved.append(str(image))
            print('D3 RealWaste images SHA-256 verified:',len(resolved),'root:',root)
            return resolved,root,image_counts
        except (FileNotFoundError,ValueError,RuntimeError) as exc:
            print('D3 candidate rejected:',root,'reason:',str(exc)[:180])
    raise RuntimeError('No RealWaste root matches the 3497 locked image hashes.')

external_paths, external_image_root, d3_counts=resolve_external(external_df,external_download)
external_df=external_df.copy(); external_df['resolved_path']=external_paths
external_eval_df=external_df.sort_values('external_relative_path').reset_index(drop=True)
assert len(external_eval_df)==3497
print('External evaluated images:',len(external_eval_df))

D3 RealWaste class-root candidates: [PosixPath('/kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste')]
D3 RealWaste images SHA-256 verified: 3497 root: /kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste
External evaluated images: 3497


## 6. Restore exact Stage 2 standalone ResNet50 architecture and preprocessing

The standalone checkpoint comes from `Resnet_50_Acc_s2.ipynb`: a wrapper `ResNet50_Model` with `self.model = torchvision.models.resnet50(...)` and a six-class `self.model.fc`. At inference we pass `weights=None`, because pretrained weights have already been absorbed into the saved checkpoint. The wrapper deliberately preserves the saved `model.*` keys and loads with `strict=True`. No training, fine-tuning, or adaptation is performed.

In [6]:
TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((350,350)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3,(0.5,)*3),
])

class ResNet50_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.resnet50(weights=None)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

def construct_resnet50():
    return ResNet50_Model(num_classes=len(CLASSES))

class LockedImageDataset(Dataset):
    def __init__(self, df, kind):
        self.df = df.reset_index(drop=True)
        self.path_col = 'external_relative_path'
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row['resolved_path']) as im:
            image = TEST_TRANSFORM(im.convert('RGB'))
        return image, int(CLASS_TO_ID[row['class']]), i

def locked_loader(df,kind):
    return DataLoader(LockedImageDataset(df,kind),batch_size=BATCH_SIZE,
                      shuffle=False,num_workers=0,pin_memory=(DEVICE.type=='cuda'))

def restore_checkpoint(seed):
    p=CHECKPOINTS[seed]
    # PyTorch weights_only=True is appropriate for the Stage 2 dictionary format.
    ckpt=torch.load(p,map_location='cpu',weights_only=True)
    assert isinstance(ckpt,dict) and 'state_dict' in ckpt, f'Unexpected checkpoint format: {p}'
    assert str(ckpt.get('model'))==CHECKPOINT_MODEL_METADATA, f'Model mismatch in {p.name}: {ckpt.get("model")}'
    assert int(ckpt.get('seed'))==seed, f'Seed mismatch: {p.name}'
    assert ckpt.get('architecture')==CHECKPOINT_ARCHITECTURE, f'Architecture mismatch: {p.name}'
    state=ckpt['state_dict']
    assert tuple(state['model.fc.weight'].shape)==(6,2048), f'Unexpected ResNet head shape in {p.name}'
    model=construct_resnet50()
    model.load_state_dict(state,strict=True)
    model.to(DEVICE).eval()
    for param in model.parameters():param.requires_grad_(False)
    return model,ckpt,file_sha256(p)

def infer(model,df,kind):
    labels, probabilities, positions = [],[],[]
    with torch.inference_mode():
        for images,y,index in locked_loader(df,kind):
            images=images.to(DEVICE,non_blocking=True)
            prob=torch.softmax(model(images),dim=1)
            probabilities.append(prob.cpu().numpy().astype(np.float64))
            labels.append(y.numpy()); positions.append(index.numpy())
    labels=np.concatenate(labels).astype(np.int64)
    probabilities=np.concatenate(probabilities)
    positions=np.concatenate(positions)
    assert np.array_equal(positions,np.arange(len(df))), 'Image order changed during inference.'
    assert np.array_equal(labels,df['class'].map(CLASS_TO_ID).to_numpy()), 'Label order mismatch.'
    assert probabilities.shape==(len(df),6) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1),1,atol=2e-6)
    return labels,probabilities

_smoke, _smoke_meta, _smoke_hash = restore_checkpoint(SEEDS[0])
with torch.inference_mode():
    batch=torch.zeros(1,3,350,350,device=DEVICE)
    assert tuple(_smoke(batch).shape)==(1,6)
print('ResNet50 strict-load and six-logit smoke test: PASS')
del _smoke,_smoke_meta,_smoke_hash,batch
if DEVICE.type=='cuda':torch.cuda.empty_cache()

ResNet50 strict-load and six-logit smoke test: PASS


## 7. Optional historical Stage 2 ResNet50 prediction references

Historical per-image prediction files are not used for the D3 evaluation. The D3-only workflow records the frozen checkpoint hashes and evaluates the locked external cohort directly.

## 8. Metrics, predictions, and confusion-matrix writers

Every metric is computed from the same fixed image ordering. Precision, recall and F1 use **macro averaging across all six classes** (`zero_division=0`). All CSV metric fields are stored as percentages.

In [7]:
METRICS=('accuracy','macro_precision','macro_recall','macro_f1')
PRED_DIR=OUTPUT/'Predictions'; PRED_DIR.mkdir(exist_ok=True)
CM_DIR=OUTPUT/'Confusion_Matrices'; CM_DIR.mkdir(exist_ok=True)

def measure(seed,kind,df,y_true,prob,checkpoint_sha,verification_status):
    y_pred=prob.argmax(axis=1).astype(int)
    precision,recall,f1,_=precision_recall_fscore_support(
        y_true,y_pred,labels=list(range(6)),average='macro',zero_division=0)
    scores={'model':MODEL_NAME,'seed':seed,'dataset':kind,'n_images':len(df),
            'accuracy':100*accuracy_score(y_true,y_pred),
            'macro_precision':100*precision,
            'macro_recall':100*recall,'macro_f1':100*f1,
            'checkpoint_sha256':checkpoint_sha,
            'external_manifest_fingerprint':EXTERNAL_FINGERPRINT}
    return scores,y_pred

def save_predictions(seed,kind,df,y_true,y_pred,prob,checkpoint_sha):
    relative='source_path' if kind=='Original463' else 'external_relative_path'
    out=pd.DataFrame({
        'model':MODEL_NAME,'seed':seed,'dataset':kind,
        'image_relative_path':df[relative].astype(str).tolist(),
        'image_sha256':df['sha256'].tolist(),
        'true_class_id':y_true,
        'true_class':[CLASSES[i] for i in y_true],
        'predicted_class_id':y_pred,
        'predicted_class':[CLASSES[i] for i in y_pred],
        'prediction_confidence':prob.max(axis=1),
        'checkpoint_sha256':checkpoint_sha})
    for i,c in enumerate(CLASSES): out[f'prob_{c}']=prob[:,i]
    assert len(out)==len(df) and out['image_relative_path'].is_unique
    destination=PRED_DIR/f'ResNet50_seed{seed}_{kind}_probabilities.csv'
    out.to_csv(destination,index=False,float_format='%.10g')
    return destination

def save_per_class(seed,kind,y_true,y_pred):
    values=classification_report(y_true,y_pred,labels=list(range(6)),target_names=list(CLASSES),
                                 output_dict=True,zero_division=0)
    records=[]
    for cls in CLASSES:
        r=values[cls]
        records.append({'model':MODEL_NAME,'seed':seed,'dataset':kind,'class':cls,
                        'output_index':CLASS_TO_ID[cls],'support':int(r['support']),
                        'precision_pct':100*r['precision'],
                        'recall_pct':100*r['recall'],'f1_pct':100*r['f1-score']})
    path=OUTPUT/f'ResNet50_seed{seed}_{kind}_per_class.csv'
    pd.DataFrame(records).to_csv(path,index=False,float_format='%.10f')
    return path

def save_confusion(seed,kind,y_true,y_pred):
    cm=confusion_matrix(y_true,y_pred,labels=list(range(6)))
    path=CM_DIR/f'ResNet50_seed{seed}_{kind}_confusion.csv'
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(path)
    fig,ax=plt.subplots(figsize=(7,6))
    ax.imshow(cm,cmap='Blues')
    ax.set_xticks(range(6),labels=CLASSES,rotation=45,ha='right')
    ax.set_yticks(range(6),labels=CLASSES)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
    ax.set_title(f'ResNet50 | seed {seed} | {kind} | n={len(y_true)}')
    threshold=cm.max()/2
    for i in range(6):
        for j in range(6):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                    color='white' if cm[i,j]>threshold else 'black',fontsize=9)
    fig.tight_layout()
    fig.savefig(CM_DIR/f'ResNet50_seed{seed}_{kind}_confusion.png',dpi=160)
    plt.close(fig)
    return path

## 8. Run five frozen seed-matched standalone ResNet50 models on the 3,497-image D3 cohort

For each seed, load the original frozen standalone ResNet50 checkpoint and evaluate the same locked D3 images. No D1 test-set inference or historical original-test comparison is performed.

In [8]:
per_seed=[]
checkpoint_registry=[]
for seed in SEEDS:
    print(f'\n=== RESNET50 SEED {seed} ===',flush=True)
    model, metadata, checkpoint_sha = restore_checkpoint(seed)
    checkpoint_registry.append({
        'model':MODEL_NAME,'seed':seed,'checkpoint_path':str(CHECKPOINTS[seed]),
        'checkpoint_sha256':checkpoint_sha,'checkpoint_split_sha256':metadata.get('split_sha256'),
        'checkpoint_best_epoch':metadata.get('best_epoch'),
        'framework':'torchvision.models.resnet50','num_output_classes':6,
        'input_size':'350x350','normalization_mean':'0.5,0.5,0.5',
        'normalization_std':'0.5,0.5,0.5','external_manifest_fingerprint':EXTERNAL_FINGERPRINT})
    y_true_ext,p_ext=infer(model,external_eval_df,'external')
    ext_score,pred_ext=measure(seed,'External3497',external_eval_df,y_true_ext,p_ext,checkpoint_sha,'NOT_APPLICABLE_D3_ONLY')
    per_seed.append(ext_score)
    save_predictions(seed,'External3497',external_eval_df,y_true_ext,pred_ext,p_ext,checkpoint_sha)
    save_per_class(seed,'External3497',y_true_ext,pred_ext)
    save_confusion(seed,'External3497',y_true_ext,pred_ext)
    print('External n:',len(y_true_ext),'| Accuracy:',round(ext_score['accuracy'],4),
          '| Macro-F1:',round(ext_score['macro_f1'],4))
    del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

results_df=pd.DataFrame(per_seed)
registry_df=pd.DataFrame(checkpoint_registry)
results_df.to_csv(OUTPUT/'ResNet50_All_Seeds_D3_External.csv',index=False,float_format='%.10f')
registry_df.to_csv(OUTPUT/'ResNet50_Checkpoint_Registry.csv',index=False)
assert len(registry_df)==5 and len(results_df)==5
assert results_df.groupby('dataset').size().to_dict()=={'External3497':5}
print('\nPER-SEED RESNET50 D3 RESULTS:')
print(results_df[['seed','dataset','n_images',*METRICS]].to_string(index=False,float_format=lambda v:f'{v:.4f}'))


=== RESNET50 SEED 7 ===
External n: 3497 | Accuracy: 34.9728 | Macro-F1: 31.1153

=== RESNET50 SEED 49 ===
External n: 3497 | Accuracy: 40.6348 | Macro-F1: 39.4106

=== RESNET50 SEED 112 ===
External n: 3497 | Accuracy: 43.5802 | Macro-F1: 39.8625

=== RESNET50 SEED 345 ===
External n: 3497 | Accuracy: 37.861 | Macro-F1: 35.1608

=== RESNET50 SEED 476 ===
External n: 3497 | Accuracy: 43.7518 | Macro-F1: 39.2519

PER-SEED RESNET50 D3 RESULTS:
 seed      dataset  n_images  accuracy  macro_precision  macro_recall  macro_f1
    7 External3497      3497   34.9728          37.4565       33.4323   31.1153
   49 External3497      3497   40.6348          42.7591       40.8255   39.4106
  112 External3497      3497   43.5802          42.6100       39.2391   39.8625
  345 External3497      3497   37.8610          43.5327       36.4575   35.1608
  476 External3497      3497   43.7518          43.5724       41.6642   39.2519


## 9. Five-seed mean ± sample standard deviation on D3

Summarizes the five independent seed-matched frozen standalone ResNet50 evaluations on the same locked 3,497-image external cohort.

In [9]:
summary=[]
for metric in METRICS:
    values=results_df.set_index('seed').loc[list(SEEDS),metric]
    summary.append({
        'model':MODEL_NAME,'metric':metric,'n_seeds':5,
        'external_images':3497,
        'external_mean_pct':values.mean(),
        'external_sample_sd_pct':values.std(ddof=1)})
summary_df=pd.DataFrame(summary)
summary_df.to_csv(OUTPUT/'ResNet50_Five_Seed_Mean_SD_D3_External.csv',
                  index=False,float_format='%.10f')
print('RESNET50 — D3 EXTERNAL (3497 images)')
for row in summary_df.itertuples(index=False):
    print(f'{row.metric:17s}: {row.external_mean_pct:.2f} +/- {row.external_sample_sd_pct:.2f}%')

RESNET50 — D3 EXTERNAL (3497 images)
accuracy         : 40.16 +/- 3.78%
macro_precision  : 41.99 +/- 2.57%
macro_recall     : 38.32 +/- 3.38%
macro_f1         : 36.96 +/- 3.78%


## 10. Export D3 predictions, metrics, and checkpoint identities

Exports five-seed D3 predictions and probabilities, per-class metrics, confusion matrices, mean ± sample SD, and SHA-256 identities of all 5 standalone ResNet50 checkpoints.

In [10]:
report={
    'experiment':'ResNet50 Stage 3 D3-only external classification evaluation',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_notebook':'Resnet_50_Acc_s2.ipynb',
    'checkpoint_source':'Stage 2 standalone ResNet50_Torchvision; best-validation-loss selected',
    'checkpoint_dataset_input':str(INPUT_ROOT),'checkpoint_seeds':list(SEEDS),
    'stage3_external_dataset':EXTERNAL_HANDLE,
    'd3_manifest_csv_path':str(EXTERNAL_MANIFEST),
    'stage3_external_dataset_version':'Kaggle RealWaste mounted copy; version unverified',
    'external_download_method':external_download_method,
    'external_resolved_path':str(external_download),
    'external_evaluation_count':3497,
    'd1_original_test_evaluation_performed':False,
    'external_locked_manifest_fingerprint_sha256':EXTERNAL_FINGERPRINT,
    'class_to_idx':CLASS_TO_ID,
    'external_transform':'RGB,Resize(350,350),ToTensor,Normalize(mean=.5x3,std=.5x3)',
    'model_constructor':'ResNet50_Model wrapper around torchvision.models.resnet50(weights=None); model.fc Linear(2048,6)',
    'model_loading':'torch.load(weights_only=True); load_state_dict(strict=True); eval; inference_mode',
    'training_or_adaptation_performed':False,
    'checkpoint_selection_on_external_dataset':False,
    'external_subset_shared_across_models':'D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv',
    'external_3497_test_sha256_verified':True,
    'same_checkpoints_for_future_energy_inference':'ResNet50_Checkpoint_Registry.csv: match standalone ResNet50 SHA-256 per seed',
    'torch':torch.__version__,
    'torchvision':__import__('torchvision').__version__,
    'device':str(DEVICE),
    'limitations':[
        'No D1 original-test results or D1-versus-D3 generalization gap are reported.',
        'Automated conservative screening does not demonstrate independent dataset acquisition.'
    ]}
(OUTPUT/'ResNet50_External_Validation_Configuration.json').write_text(
    json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')

archive=OUTPUT/'ResNet50_D3_Standalone_External_Only_Outputs.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as outzip:
    for f in sorted(OUTPUT.rglob('*')):
        if f.is_file() and f!=archive and 'original_source_extracted' not in f.parts:
            outzip.write(f,f.relative_to(OUTPUT))
print('OUTPUTS SAVED:',OUTPUT)
print('ZIP:',archive)
print('STAGE 3 D3-ONLY RESNET50 CLASSIFICATION EVALUATION COMPLETE: True')

OUTPUTS SAVED: /kaggle/working/ResNet50_D3_Standalone_External_Validation
ZIP: /kaggle/working/ResNet50_D3_Standalone_External_Validation/ResNet50_D3_Standalone_External_Only_Outputs.zip
STAGE 3 D3-ONLY RESNET50 CLASSIFICATION EVALUATION COMPLETE: True
